<a href="https://colab.research.google.com/github/TingyuZhao19/MSSP6070/blob/main/WeeklyModules/Week09/PARTICIPATION_ACTIVITY_Week_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Define Search Parameters

The subtask requires defining the image category, output directory, and an empty dictionary for headers. This code block will initialize these variables as per the instructions.


In [12]:
search_query = 'cats'
output_directory = 'imgur_images'
headers = {}

print(f"Search Query: {search_query}")
print(f"Output Directory: {output_directory}")
print(f"Headers: {headers}")

Search Query: cats
Output Directory: imgur_images
Headers: {}


## Scrape Image URLs

Implement code to access Imgur, search for the specified category, and extract the direct URLs of the images. This will likely involve using `requests` for HTTP requests and potentially a parsing library like `BeautifulSoup` if scraping HTML, or handling JSON responses.


In [13]:
import requests
from bs4 import BeautifulSoup
import os

# Construct the Imgur search URL
imgur_url = f"https://imgur.com/search?q={search_query}"
print(f"Searching Imgur for: {search_query} at {imgur_url}")

# Make the HTTP GET request
try:
    response = requests.get(imgur_url, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors
    print("Successfully fetched Imgur search page.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching Imgur search page: {e}")
    response = None

# Parse the content if the request was successful
soup = None
if response:
    soup = BeautifulSoup(response.content, 'html.parser')
    print("Successfully parsed HTML content.")

Searching Imgur for: cats at https://imgur.com/search?q=cats
Error fetching Imgur search page: 429 Client Error: Unknown Error for url: https://imgur.com/search?q=cats


The previous attempt to fetch the Imgur page resulted in a 429 Client Error, likely due to rate-limiting or missing user-agent headers. To address this, I will add a common User-Agent header to the `headers` dictionary to make the request appear more like a legitimate browser request, and then re-attempt the request to fetch and parse the Imgur search page.



In [14]:
import requests
from bs4 import BeautifulSoup
import os

# Add User-Agent header to mimic a web browser and avoid 429 errors
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Construct the Imgur search URL
imgur_url = f"https://imgur.com/search?q={search_query}"
print(f"Searching Imgur for: {search_query} at {imgur_url}")

# Make the HTTP GET request
try:
    response = requests.get(imgur_url, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors
    print("Successfully fetched Imgur search page.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching Imgur search page: {e}")
    response = None

# Parse the content if the request was successful
soup = None
if response:
    soup = BeautifulSoup(response.content, 'html.parser')
    print("Successfully parsed HTML content.")

Searching Imgur for: cats at https://imgur.com/search?q=cats
Successfully fetched Imgur search page.
Successfully parsed HTML content.


Now the next step is to inspect the HTML structure to identify the elements containing image URLs. Imgur often stores image sources in specific data attributes or within `<img>` tags. I will look for common patterns to extract these URLs.



In [15]:
image_urls = set() # Use a set to store unique URLs

# Find direct <img> tags with src attributes
for img in soup.find_all('img', src=True):
    src = img['src']
    if src.startswith('//'): # Handle protocol-relative URLs
        src = 'https:' + src
    if src.startswith('http') and ('.jpg' in src or '.png' in src or '.gif' in src):
        image_urls.add(src)

# Imgur also often uses a data-src or other custom attributes for lazy loading or different resolutions.
# We need to inspect the page to find the most reliable attribute.
# For a common Imgur search page, images are often in 'div' elements with data-src attributes

# Example for common Imgur image structure
for gallery_item in soup.find_all('div', class_='image-list-link') or soup.find_all('a', class_='image-list-link'):
    # Try to find an img tag within the item
    img_tag = gallery_item.find('img')
    if img_tag and 'src' in img_tag.attrs:
        src = img_tag['src']
        if src.startswith('//'): # Handle protocol-relative URLs
            src = 'https:' + src
        if src.startswith('http') and ('.jpg' in src or '.png' in src or '.gif' in src):
            image_urls.add(src)
    elif 'data-src' in gallery_item.attrs: # Sometimes the image URL is directly on the parent div/a
        src = gallery_item['data-src']
        if src.startswith('//'): # Handle protocol-relative URLs
            src = 'https:' + src
        if src.startswith('http') and ('.jpg' in src or '.png' in src or '.gif' in src):
            image_urls.add(src)


print(f"Found {len(image_urls)} unique image URLs.")

Found 53 unique image URLs.


## Download Images

### Subtask:
Iterate through the collected image URLs, download each image, and save it to the specified local directory. Implement error handling for cases where an image cannot be downloaded.


**Reasoning**:
I will create a code block that iterates through the collected image URLs, downloads each image, saves it to the specified local directory, and includes error handling as per the instructions.



In [16]:
import os
import requests

# Ensure the output directory exists
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
    print(f"Created output directory: {output_directory}")
else:
    print(f"Output directory already exists: {output_directory}")

downloaded_count = 0
failed_count = 0

print(f"Attempting to download {len(image_urls)} images...")

for url in image_urls:
    try:
        # Extract filename from URL
        filename = os.path.basename(url.split('?')[0]) # Remove query parameters if any
        if not filename:
            # Handle cases where URL ends with a slash or doesn't have a clear filename
            filename = 'image_' + str(hash(url)) + '.jpg' # Simple unique name

        filepath = os.path.join(output_directory, filename)

        # Download the image
        response = requests.get(url, headers=headers, stream=True, timeout=10)
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)

        # Save the image content
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f"Successfully downloaded: {filename}")
        downloaded_count += 1

    except requests.exceptions.HTTPError as e:
        print(f"Error downloading {url} (HTTP Error): {e}")
        failed_count += 1
    except requests.exceptions.ConnectionError as e:
        print(f"Error downloading {url} (Connection Error): {e}")
        failed_count += 1
    except requests.exceptions.Timeout as e:
        print(f"Error downloading {url} (Timeout Error): {e}")
        failed_count += 1
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url} (Other Request Error): {e}")
        failed_count += 1
    except IOError as e:
        print(f"Error saving {filename} to {output_directory} (IO Error): {e}")
        failed_count += 1
    except Exception as e:
        print(f"An unexpected error occurred for {url}: {e}")
        failed_count += 1

print(f"\n--- Download Summary ---")
print(f"Total images to attempt: {len(image_urls)}")
print(f"Successfully downloaded: {downloaded_count}")
print(f"Failed to download: {failed_count}")

Output directory already exists: imgur_images
Attempting to download 53 images...
Successfully downloaded: tU50Ek2b.jpg
Successfully downloaded: XbqaYVXb.jpg
Successfully downloaded: bLIR0Mhb.jpg
Successfully downloaded: EpHpQ1tb.jpg
Successfully downloaded: QxR8tQhb.jpg
Successfully downloaded: 6qL2HSNb.jpg
Successfully downloaded: Ejn0Yvib.jpg
Successfully downloaded: U0iADj9b.jpg
Successfully downloaded: ckGhJXYb.jpg
Successfully downloaded: wYTCtRub.jpg
Successfully downloaded: i8Wc70sb.jpg
Successfully downloaded: zKt1JF0b.jpg
Successfully downloaded: tf5hu9Rb.jpg
Successfully downloaded: YZ2Vtmrb.jpg
Successfully downloaded: ose0MfDb.jpg
Successfully downloaded: UEvlZERb.jpg
Successfully downloaded: 43ESX4cb.jpg
Successfully downloaded: grXqcNwb.jpg
Successfully downloaded: pbao8mhb.jpg
Successfully downloaded: Pmyf38cb.jpg
Successfully downloaded: icsm6L3b.jpg
Successfully downloaded: KWvtdg0b.jpg
Successfully downloaded: OJTwZucb.jpg
Successfully downloaded: dRxnay8b.jpg
Succes